<a href="https://colab.research.google.com/github/Benswe/Building-Micrograd/blob/main/BuildingGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-08-11 22:45:58--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-08-11 22:45:58 (34.3 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [4]:
len(text)

1115394

In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


Tokenize into integers

In [6]:
stoi = {s: i for i, s in enumerate(chars)}
itos = {i: s for i,s in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hii there'))
print(decode(encode(('hii there'))))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [7]:
data = torch.tensor(encode(text), dtype=torch.long)
data.shape, data.dtype

(torch.Size([1115394]), torch.int64)

In [8]:
# split into train and validation sets
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
context = 8
train_data[:context+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

time dimension:

In [10]:
x = train_data[:context]
y = train_data[1:context+1]
for t in range(context):
  context = x[:t+1]
  target= y[t]
  print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


batch dimension for efficiency:

In [12]:
device = 'cuda' if torch.accelerator.is_available() else 'cpu'

In [13]:
torch.manual_seed(1337)
batch_size = 4
context = 8

def get_batch(split):
  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data) - context, (batch_size, ))
  x = torch.stack([data[i: i+context] for i in ix])
  y = torch.stack([data[i+1:i+context+1] for i in ix])
  # move to gpu
  x, y = x.to(device), y.to(device)
  return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0')


In [14]:
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    # each token directly reads off the logits for the next token from a lookup table

    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
  def forward(self, idx, targets=None):
    # idx and targets are both (B, T) tensors of integers
    logits = self.token_embedding_table(idx) # (B, T, C)
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(-1)
      loss = F.cross_entropy(logits, targets)
    return logits, loss
  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      # get predictions
      logits, loss = self.forward(idx)
      # take the last time step
      logits = logits[:, -1, :]
      probs = F.softmax(logits, dim=1)

      # sample from distribution
      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [17]:
model = BigramLanguageModel(vocab_size)

In [18]:
# move to GPU
model = model.to(device)

In [19]:
# create a pytorch optimizer, use high learning rate cause small model
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [20]:
@torch.no_grad() # for efficency
def estimate_loss():
  out = {}
  model.eval()

  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)

      logits, loss = model(X, Y)
      losses[k] = loss.item()

    out[split] = losses.mean()
  model.train()
  return out



In [21]:
# training updates
num_iters = 100000
eval_iters = 200
for iter in range(num_iters):
  # eval the loss once in a while
  if iter % 10000 == 0:
    losses = estimate_loss()
    print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")


  xb, yb = get_batch("train")
  logits, loss = model(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()



step 0: train loss 4.7303, val loss 4.7214
step 10000: train loss 2.5143, val loss 2.5607
step 20000: train loss 2.4378, val loss 2.5058
step 30000: train loss 2.4487, val loss 2.4874
step 40000: train loss 2.4570, val loss 2.4497
step 50000: train loss 2.4455, val loss 2.4629
step 60000: train loss 2.4644, val loss 2.4830
step 70000: train loss 2.4832, val loss 2.4999
step 80000: train loss 2.4470, val loss 2.5280
step 90000: train loss 2.4634, val loss 2.4612


In [22]:
# generate from the model
idx = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(idx=idx, max_new_tokens=100)[0].tolist()))




CExthy brid owindakis by bth

Hiset bube d e.
S:
O:
IS:
Falatanss:
Wanthar usqur, vet?
F dilasoate
